# Week 39

In [7]:
# !pip install -q --upgrade transformers datasets sacrebleu

In [8]:
import pandas as pd
import re
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import nltk
from nltk.util import ngrams
import math
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk import FreqDist, ConditionalFreqDist
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch import nn
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from typing import List, Tuple
from transformers import MarianMTModel, MarianTokenizer, DistilBertTokenizerFast
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from tqdm import tqdm
import pandas as pd

## Load the dataset
splits = {'train': 'train.parquet', 'validation': 'validation.parquet'}
df_train = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["train"])
df_val = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["validation"])

# Only keep Arabic, Telugu and Korean examples
df_train = df_train[df_train['lang'].isin(['ar', 'te', 'ko'])]

df_train_te = df_train[df_train['lang'].isin(['te'])]

df_val_te = df_val[df_val['lang'].isin(['te'])]


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\CoolD\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\CoolD\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [9]:
# Load the model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small")

c:\Users\CoolD\miniconda3\envs\NLP\Lib\site-packages\transformers\convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm
import pandas as pd

# --- Device ---
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Load multilingual mBART50 model ---
model_name = "facebook/mbart-large-50-many-to-many-mmt"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

In [11]:
# --- Language mapping ---
LANG_MAP = {
    "te": "te_IN",   # Telugu
    "ar": "ar_AR",   # Arabic
    "ko": "ko_KR"    # Korean
}
TARGET_LANG = "te_IN"  # Telugu

# --- Translation function (English → Telugu) ---
def translate_to_telugu_mbart(text, source_lang="en_XX"):
    tokenizer.src_lang = source_lang  # English input
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)

    translated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.lang_code_to_id[TARGET_LANG],
        max_length=64,
        num_beams=5,
        early_stopping=True
    )

    translated_text = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
    return translated_text

# --- Example DataFrame ---
# df_train_te should have columns: ['question', 'context', 'answer']
# where 'answer' is in English
df_train_te = df_train_te[df_train_te['answer'].notnull()].reset_index(drop=True)

# Translate English answers → Telugu answers
tqdm.pandas(desc="Translating English answers to Telugu")
df_train_te['answer_inlang_trans'] = df_train_te['answer'].progress_apply(
    lambda x: translate_to_telugu_mbart(x)
)

# Check some examples
print(df_train_te[['question', 'answer', 'answer_inlang_trans','answerable']].head())

df_train_te


Translating English answers to Telugu:   0%|          | 0/1355 [00:00<?, ?it/s]Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Translating English answers to Telugu:   0%|          | 6/1355 [00:27<1:44:26,  4.65s/it]


KeyboardInterrupt: 

In [ ]:
df_train_te

,question,context,lang,answerable,answer_start,answer,answer_inlang,answer_inlang_trans
0,ప్రపంచంలో మొట్టమొదటి దూర విద్య విద్యాలయం ఏ దే...,"Referred to as ""People's University"" by Charle...",te,True,236,London,None,లండన్
1,1959వ సంవత్సరంలో భారతదేశ ప్రధాన మంత్రి ఎవరు?,"Since 1947, there have been 14 different prime...",te,True,220,Jawaharlal Nehru,None,Jawaharlal Nehru
2,ఏ కాకతీయ రాజు కర్నూలు జిల్లాను చివరిగా పాలించాడు?,"Rani Rudrama Devi (died 1289 or 1295), who def...",te,True,194,Prataparudra,None,Prataparudra
3,మానవ హక్కులు ఎన్ని?,The Declaration consists of 30 articles affirm...,te,True,28,30,None,30
4,భారదేశంలో అత్యధిక జనాభా కలిగిన రాష్ట్రం ఏది?,"Uttar Pradesh (; IAST: ""Uttar Pradeś"" ) is a s...",te,True,0,Uttar Pradesh,None,ઉત્તર પ્રદેશ
...,...,...,...,...,...,...,...,...
1350,కోళ్లు ఎక్కువగా ఏ దేశంలో కనిపిస్తాయి?,"Since time immemorial, man has been practicing...",te,False,-1,United States of America,అమెరికా సంయుక్త రాష్ట్రాలు,అమెరికా యునైటెడ్ స్టేట్స్
1351,క్షయ వ్యాధికి విరుగుడు ఏ దేశంలో కనుగొన్నారు?,Vaccines against anthrax for use in livestock ...,te,False,-1,France,ఫ్రాన్స్,ఫ్రెంచ్
1352,ఖురాన్ ఏ అరబ్బీ భాషలో ఎవరు రాసారు?,are broken Other Names of the Qur'an: It is be...,te,False,-1,Prophet Muhammad,ముహమ్మద్ ప్రవక్త,പ്രവാചകం മുഹമ്മദ്
1353,టెక్సస్ రాష్ట్రంలోని అతిపెద్ద మానవ నిర్మితం ఏది ?,Austin is the capital of the US state of Texas...,te,False,-1,JP Morgan Chase Tower,జేపీ మోర్గాన్ ఛేజ్ టవర్,JPమోర్గాన్ చైస్ టవర్


In [ ]:
df_train_te['answer_inlang'] = df_train_te['answer_inlang'].combine_first(df_train_te['answer_inlang_trans'])

# Optional: drop the temporary column
df_train_te = df_train_te.drop(columns=['answer_inlang_trans'])

## Question + Context

In [ ]:
# Get the question and context seperated by [SEP] token
def get_question_context(row):
    question = row['question']
    context = row['context']
    return question + " [SEP] " + context

df_train_te['input_text'] = df_train_te.apply(get_question_context, axis=1)
df_val_te['input_text'] = df_val_te.apply(get_question_context, axis=1)

df_train_te = df_train_te[['input_text', 'answer', 'answer_inlang','answerable']]
df_val_te = df_val_te[['input_text', 'answer', 'answer_inlang','answerable']]


/tmp/ipython-input-3479198400.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_val_te['input_text'] = df_val_te.apply(get_question_context, axis=1)


In [ ]:
from transformers import DataCollatorForSeq2Seq
from datasets import Dataset

# Make sure the column exists
df_train_te = df_train_te[df_train_te['answer_inlang'].notnull()]
df_train_te = df_train_te[['input_text', 'answer_inlang']].rename(columns={'answer_inlang': 'target_text'})

df_val_te = df_val_te[df_val_te['answer_inlang'].notnull()]
df_val_te = df_val_te[['input_text', 'answer_inlang']].rename(columns={'answer_inlang': 'target_text'})

# Convert to HF Dataset
train_dataset = Dataset.from_pandas(df_train_te, preserve_index=False)
val_dataset = Dataset.from_pandas(df_val_te, preserve_index=False)

max_input_length = 512   # depending on context length
max_target_length = 64   # answers are short

def tokenize_function(examples):
    # Tokenize inputs (question + context)
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=max_input_length,
        truncation=True
    )

    # Tokenize targets (Telugu answers)
    labels = tokenizer(
        examples["target_text"],
        max_length=max_target_length,
        truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

train_dataset = Dataset.from_pandas(df_train_te)
val_dataset = Dataset.from_pandas(df_val_te)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)


Map:   0%|          | 0/1355 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

import evaluate
metric = evaluate.load("sacrebleu")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace -100 in labels with pad_token_id
    labels = [[(l if l != -100 else tokenizer.pad_token_id) for l in label] for label in labels]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # sacreBLEU expects list of references per prediction
    result = metric.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    return {"bleu": result["score"]}


In [ ]:
import transformers
print(transformers.__version__)  # should be >= 4.44.2 (or at least > 4.20)


4.56.2


In [ ]:
import transformers
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5-te-qa",
    # evaluation_strategy="epoch",
    # evaluate_during_training=True,
    label_smoothing_factor=0.1,
    learning_rate=5e-6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=5,
    predict_with_generate=True,
    logging_dir="./logs",
    logging_steps=50,
    save_safetensors=False
)


In [ ]:
from transformers import Seq2SeqTrainer


n = 100
shuffled = tokenized_train.shuffle(seed=42)

tokenized_train = shuffled.select(range(n))

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

/tmp/ipython-input-4203548990.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Step,Training Loss
50,2.216900
100,1.873400
150,1.870400
200,1.825100
250,1.784800
300,1.777000
350,1.760500
400,1.758400
450,1.739200


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

for example in tokenized_val.select(range(20)):
    input_text = example['input_text']
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512
    )

    # Move input tensors to same device as model
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Generate
    outputs = model.generate(**inputs, max_length=64, num_beams=5, early_stopping=True)

    print("Question + Context:", input_text)
    print("Generated Answer:", tokenizer.decode(outputs[0], skip_special_tokens=True))
    print("Reference:", example['target_text'])
    print("-----")


In [ ]:
# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

test_q = "బంగ్లాదేశ్ దేశ జాతీయ భాష ఏది"  # Example Telugu question
test_ctx = "English is a co-official language of Bangladesh and is widely used in the executive, legislative and judicial branches. Bangladesh's Constitution and laws were written in English and are now being re-written in the Bengali. It is also widely used in schools, colleges and universities as a medium of instruction"
input_text = test_q + " [SEP] " + test_ctx

inputs = tokenizer(input_text, return_tensors="pt", truncation=True, padding=True)
inputs = {k: v.to(device) for k, v in inputs.items()}  # move tensors to same device
outputs = model.generate(**inputs, max_length=64)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


In [ ]:
results = trainer.evaluate()

print(results)

df_val_te

NameError: name 'trainer' is not defined

In [ ]:
val_answerable = tokenized_val.filter(lambda x: x["answerable"] == True)
val_unanswerable = tokenized_val.filter(lambda x: x["answerable"] == False)

results_answerable = trainer.evaluate(eval_dataset=val_answerable)
results_unanswerable = trainer.evaluate(eval_dataset=val_unanswerable)

print("Answerable:", results_answerable)
print("Unanswerable:", results_unanswerable)
